# Types of RNN in Keras
The same SimpleRNN layer gives many-to-one, many-to-many, one-to-many and encoder-decoder models.
We build each one untrained and print the input and output shapes.

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"      # hide TensorFlow's start-up messages
from pathlib import Path
import numpy as np
import pandas as pd
import keras
import tensorflow as tf

keras.utils.set_random_seed(42)
tf.config.experimental.enable_op_determinism()   # same numbers on every run
DATA = Path("data")
DATA.mkdir(exist_ok=True)

## The data: one-hot reviews from the RNN forward propagation Note
3 reviews, padded to 4 time steps, 5 input features.

In [ ]:
vocab = ["movie", "was", "good", "bad", "not"]
one_hot = {w: np.eye(5)[i] for i, w in enumerate(vocab)}
reviews = ["movie was good", "movie was bad", "movie was not good"]
X = keras.utils.pad_sequences([np.array([one_hot[w] for w in r.split()]) for r in reviews],
                              dtype="float32", padding="post")
print("input shape:", X.shape)          # (batch, time steps, features)
rows = []


def show(name, model, x):
    """Print and record the input and output shapes of a model."""
    out = model(x).numpy()
    print(f"{name:32} input {x.shape} -> output {out.shape}")
    rows.append(dict(type=name, input_shape=str(x.shape), output_shape=str(out.shape)))

## Many-to-one: a sequence in, one number out
`return_sequences=False` (the default): the layer returns only the last hidden state.

In [ ]:
many_to_one = keras.Sequential([keras.Input(shape=(4, 5)),
                                keras.layers.SimpleRNN(3),
                                keras.layers.Dense(1, activation="sigmoid")])
show("many-to-one", many_to_one, X)

## Many-to-many, same length: one output per time step
`return_sequences=True`: the layer returns the hidden state of every time step.
Dense on a 3D input acts on the last axis, so it predicts at every time step
(here 4 tags per word, as in parts-of-speech tagging).

In [ ]:
many_to_many = keras.Sequential([keras.Input(shape=(4, 5)),
                                 keras.layers.SimpleRNN(3, return_sequences=True),
                                 keras.layers.Dense(4, activation="softmax")])
show("many-to-many (same length)", many_to_many, X)

In [ ]:
# The last of the returned hidden states is exactly what return_sequences=False gives
rnn_all = keras.layers.SimpleRNN(3, return_sequences=True)
rnn_last = keras.layers.SimpleRNN(3)
rnn_all.build((None, 4, 5)); rnn_last.build((None, 4, 5))
rnn_last.set_weights(rnn_all.get_weights())
all_states, last_state = rnn_all(X).numpy(), rnn_last(X).numpy()
print("all hidden states:", all_states.shape, " last hidden state:", last_state.shape)
assert np.allclose(all_states[:, -1, :], last_state, atol=1e-6)
print("last time step of return_sequences=True equals return_sequences=False")

## One-to-many: one vector in, a sequence out
A non-sequential input (say an image's 8 numbers) is repeated at 6 time steps with RepeatVector,
then the RNN returns a hidden state at each step and Dense turns each one into a word out of 5.

In [ ]:
image_features = np.random.default_rng(0).random((2, 8)).astype("float32")   # 2 "images"
one_to_many = keras.Sequential([keras.Input(shape=(8,)),
                                keras.layers.RepeatVector(6),
                                keras.layers.SimpleRNN(3, return_sequences=True),
                                keras.layers.Dense(5, activation="softmax")])
show("one-to-many", one_to_many, image_features)

## Many-to-many, different lengths: encoder-decoder
The encoder reads all 4 input words and keeps only its last hidden state.
The decoder repeats that summary at 6 output steps and produces 6 words out of a 7-word vocabulary.

In [ ]:
encoder_decoder = keras.Sequential([keras.Input(shape=(4, 5)),
                                    keras.layers.SimpleRNN(3),                          # encoder
                                    keras.layers.RepeatVector(6),
                                    keras.layers.SimpleRNN(3, return_sequences=True),   # decoder
                                    keras.layers.Dense(7, activation="softmax")])
show("many-to-many (different lengths)", encoder_decoder, X)

## One-to-one: no sequence at all
A plain network: no time steps, no recurrence.

In [ ]:
one_to_one = keras.Sequential([keras.Input(shape=(8,)),
                               keras.layers.Dense(3, activation="relu"),
                               keras.layers.Dense(1, activation="sigmoid")])
show("one-to-one", one_to_one, image_features)
pd.DataFrame(rows).to_csv(DATA / "shapes.csv", index=False)